# 🧠 CASCADES v9 Pro — Training Notebook

**Continual Adaptation via Shared Dynamic Subspaces with Bio-Inspired Sleep Consolidation**

This notebook runs the full CASCADES v9 training pipeline on Colab's GPU.

Features:
- Sequential continual learning across 3 task domains (Logic → Architecture → Code)
- Bio-inspired Sleep Consolidation between tasks
- Generative Exact Match evaluation
- 4-bit NF4 quantization (fits in T4 16GB easily)

**Runtime:** ~30-60 min on T4, ~15-30 min on A100

## 1️⃣ Setup — Install Dependencies & Clone Repo

In [ ]:
# Check GPU availability
!nvidia-smi
import torch
print(f"\nCUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info(0)
    print(f"VRAM: {total/1024**3:.1f} GB total, {free/1024**3:.1f} GB free")

In [ ]:
# Install dependencies
!pip install -q transformers>=4.44 accelerate>=0.34 bitsandbytes>=0.43 peft>=0.12 pandas
print("Dependencies installed.")

In [ ]:
# Clone the CASCADES repo
import os
REPO_URL = "https://github.com/Bender1011001/CASCADES--continual-PEFT-for-Local-LLMs.git"
REPO_DIR = "/content/CASCADES"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print(f"Cloned to {REPO_DIR}")
else:
    !cd {REPO_DIR} && git pull
    print(f"Updated {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Quick sanity check — run the sleep cycle tests
!python -m pytest tests/test_sleep.py -v --tb=short

## 2️⃣ Configuration

In [ ]:
# Training configuration
# Adjust these based on your Colab GPU tier

MODEL_ID = "p-e-w/Qwen3-4B-Instruct-2507-heretic"  # 4B parameter model
OUTPUT_PREFIX = "cascades_v9_colab"
SEED = 42
DMOLE_THRESHOLD = 0.22
EPOCHS = 3          # 3 epochs per task
EVAL_EM = True       # Run generative exact match evaluation
ENABLE_SLEEP = True  # Bio-inspired sleep consolidation

print("Configuration:")
print(f"  Model: {MODEL_ID}")
print(f"  Epochs: {EPOCHS}")
print(f"  Sleep: {'Enabled' if ENABLE_SLEEP else 'Disabled'}")
print(f"  Generative EM: {'Enabled' if EVAL_EM else 'Disabled'}")

## 3️⃣ Training — Full CASCADES Pipeline

In [ ]:
%%time

import sys
sys.path.insert(0, '/content/CASCADES')

# Import and run the training pipeline
from cascades_exp.hf_cascades_reasoning import train_cascades_v3

train_cascades_v3(
    seed=SEED,
    dmole_threshold=DMOLE_THRESHOLD,
    model_id=MODEL_ID,
    output_prefix=OUTPUT_PREFIX,
    epochs=EPOCHS,
    eval_em=EVAL_EM,
    enable_sleep=ENABLE_SLEEP,
)

## 4️⃣ Results Analysis

In [ ]:
import pandas as pd

# Load and display the accuracy matrix
results_csv = f"{OUTPUT_PREFIX}_results.csv"
df = pd.read_csv(results_csv, index_col=0)

print("Accuracy Matrix (Proxy ACC %):")
print((df * 100).round(2))
print()

# Compute BWT
import numpy as np
num_tasks = len(df)
final_accs = df.iloc[-1].values
avg_acc = np.mean(final_accs)

bwt_list = []
for i in range(num_tasks - 1):
    bwt_list.append(df.iloc[-1, i] - df.iloc[i, i])
bwt = np.mean(bwt_list)

print(f"Average Proxy Accuracy: {avg_acc * 100:.2f}%")
print(f"Backward Transfer (BWT): {bwt * 100:.2f}%")
print(f"{'POSITIVE BWT ✅' if bwt > 0 else 'NEGATIVE BWT ⚠️'} — {'No catastrophic forgetting!' if bwt >= 0 else 'Some forgetting detected.'}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Heatmap of accuracy matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df * 100, annot=True, fmt='.1f', cmap='YlGnBu',
            xticklabels=[f'Eval T{i}' for i in range(num_tasks)],
            yticklabels=[f'After T{i}' for i in range(num_tasks)],
            ax=ax, vmin=0, vmax=100)
ax.set_title(f'CASCADES v9 Proxy Accuracy Matrix\nAvg ACC: {avg_acc*100:.1f}% | BWT: {bwt*100:.2f}%')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PREFIX}_heatmap.png', dpi=150)
plt.show()

## 5️⃣ Download Results

In [ ]:
from google.colab import files

# Download the trained weights and results
weights_path = f"{OUTPUT_PREFIX}_weights.pt"
results_path = f"{OUTPUT_PREFIX}_results.csv"
heatmap_path = f"{OUTPUT_PREFIX}_heatmap.png"

for path in [weights_path, results_path, heatmap_path]:
    if os.path.exists(path):
        print(f"Downloading: {path} ({os.path.getsize(path)/1024:.1f} KB)")
        files.download(path)
    else:
        print(f"Not found: {path}")

## 6️⃣ (Optional) Load Weights Locally on 8GB GPU

After downloading the weights, you can load them on your local 4060 Ti for inference:

```python
# On your local machine:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "p-e-w/Qwen3-4B-Instruct-2507-heretic"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load CASCADES adapter weights
adapter_state = torch.load("cascades_v9_colab_weights.pt", weights_only=True)
model.load_state_dict(adapter_state, strict=False)
print(f"Loaded {len(adapter_state)} adapter parameters")
```